In [1]:
# We are going to create and train a self_supervised PatchTST model, and then fine-tune it using supervised learning.
# The model is based on the paper "Self-Supervised Learning of Patch Transformers for Time Series Classification" by Xu et al. (2022).

# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from torch.autograd import Variable

import random
import os
import numpy as np
import pandas as pd
import math
import sys
from collections import Counter
from itertools import chain
from typing import List
import textwrap

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

#from google.colab import drive

random.seed(0)
torch.manual_seed(0)

from PatchTST_self_supervised import PatchTSTSelfSupervised, self_supervised_loss
from PatchTST import PatchTST
from utils import train, val, train_self_supervised, val_self_supervised

In [2]:
%load_ext autoreload
%autoreload 1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(F"Device set to {device}")

Device set to cpu


In [3]:
from data_loader import TimeSeriesDataset, SelfSupervisedTimeSeriesDataset
input_length    = 336   # e.g. past 336 steps
forecast_horizon= 96    # e.g. next 96 steps
batch_size      = 32

dataset = SelfSupervisedTimeSeriesDataset("data/ETTh1.csv", input_length)
n_total = len(dataset)
n_train = int(0.8 * n_total)
train_idx = list(range(0, n_train))
val_idx   = list(range(n_train, n_total))

train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False
)


In [ ]:
input_length     = 336   # e.g. past 336 steps
forecast_horizon = 96    # e.g. next 96 steps
batch_size       = 32

criterion = nn.MSELoss()
model = PatchTSTSelfSupervised(input_length=input_length, patch_len=16,
                     stride=16)
optimizer = optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
print('here')
train_loss_arr, val_loss_arr = train_self_supervised(
    model, train_loader, val_loader,
    self_supervised_loss=self_supervised_loss, epochs=1,
    optimizer=optimizer, device=device
)

# Get the state_dict of the model's encoder
transformer_state_dict = model.transformer.state_dict()

train:   0%|          | 0/427 [00:00<?, ?it/s]

here
Starting training...
Epoch 1/1


train: 100%|██████████| 427/427 [08:53<00:00,  1.25s/it]


Epoch 1, Train Loss: 0.7084, Val Loss = 0.4594
Training finished.


AttributeError: 'PatchTSTSelfSupervised' object has no attribute 'encoder'

In [13]:
nput_length    = 512   # e.g. past 336 steps
forecast_horizon= 96    # e.g. next 96 steps
                        # The paper uses {96,192,336,720}
batch_size      = 32
patch_length    = 12

dataset = TimeSeriesDataset("data/ETTh1.csv", input_length, forecast_horizon)
n_total = len(dataset)
n_train = int(0.8 * n_total)
train_idx = list(range(0, n_train))
val_idx   = list(range(n_train, n_total))

train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False
)

In [14]:
# Create a new PatchTST model for supervised training
supervised_model = PatchTST(
    input_length=input_length,
    patch_len=16,
    stride=16,
    forecast_horizon=forecast_horizon
)
# Load the encoder state_dict into the new model
supervised_model.transformer.load_state_dict(transformer_state_dict)

# Do not freeze the encoder parameters
# supervised_model.encoder.requires_grad = True
fine_tune_train_loss_arr, fine_tune_val_loss_arr = train(
    supervised_model, train_loader, val_loader,
    criterion=criterion, epochs=1,
    optimizer=optimizer, device=device
)

train:   0%|          | 0/424 [00:00<?, ?it/s]

Starting training...
Epoch 1/1


train:   3%|▎         | 13/424 [00:16<08:50,  1.29s/it]


KeyboardInterrupt: 